# Understanding Checkpoint Architecture

# HydraNet Models - Best Checkpoints

In [1]:
# Analyze checkpoint to determine model configuration
import sys
sys.path.insert(0, '..')
from src import hydranet
import torch

# Download and inspect checkpoint
weight_paths = hydranet.get_model_weights(
    training='finetuning',
    model='student',
    task='anomaly_detection',
    n_shots=100,
    download_dir='../weights',
    latest_only=True,
)

state_dict = torch.load(weight_paths[0], map_location='cpu')

# Extract channel dimensions from checkpoint
channels = []
for i in range(10):  # Check up to 10 encoder layers
    key = f'encoders.{i}.channel_proj.weight'
    if key in state_dict:
        out_channels = state_dict[key].shape[0]
        channels.append(out_channels)
        print(f"Encoder {i}: {out_channels} channels")
    else:
        break

# Check bottleneck
bottleneck_key = 'bottleneck.channel_proj.weight'
if bottleneck_key in state_dict:
    bottleneck_channels = state_dict[bottleneck_key].shape[0]
    channels.append(bottleneck_channels)
    print(f"Bottleneck: {bottleneck_channels} channels")

# Determine base_filters and channel_multipliers
if channels:
    base_filters = channels[0]
    channel_multipliers = [c // base_filters for c in channels]
    print(f"\nDerived config:")
    print(f"  base_filters: {base_filters}")
    print(f"  channel_multipliers: {channel_multipliers}")
    print(f"  depth: {len(channels) - 1}")

Found latest checkpoint from 20260108
  Downloading: UNet_Myriad2_Downstream_unfrozen_best.pt
  Saved to: ../weights/finetuning/hydranet/anomaly_detection_nshot100_unfrozen/anomaly_detection/20260108_UNet_Myriad2_Downstream_unfrozen_100/UNet_Myriad2_Downstream_unfrozen_best.pt
Encoder 0: 16 channels
Encoder 1: 32 channels
Encoder 2: 64 channels
Bottleneck: 128 channels

Derived config:
  base_filters: 16
  channel_multipliers: [1, 2, 4, 8]
  depth: 3


/var/folders/21/4frhlz5x0tb46dk9rnx9jw4884jv22/T/ipykernel_87967/3379922285.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(weight_paths[0], map

In [2]:
from huggingface_hub import list_repo_files
import pandas as pd
import re

# Fetch all files from HuggingFace dataset
HF_REPO = "sirbastiano94/hydranet"
files = list_repo_files(repo_id=HF_REPO, repo_type="dataset")

# Filter for best checkpoints only
pt_files = [f for f in files if f.endswith('_best.pt')]

# Parse path structure: {training}/{model}/{task_nshotXXX}/...
def parse_path(file_path):
    parts = file_path.split('/')
    training = parts[0] if len(parts) > 0 else None
    model_raw = parts[1] if len(parts) > 1 else None
    
    # Apply naming convention: hydranet -> student, phisatnet -> teacher
    model_map = {'hydranet': 'student', 'phisatnet': 'teacher'}
    model = model_map.get(model_raw, model_raw)
    
    task, n_shots = None, None
    if len(parts) > 2:
        match = re.search(r'^(.+)_nshot(\d+)', parts[2])
        if match:
            task = match.group(1)
            n_shots = int(match.group(2))
    
    # Extract datetime from path (e.g., 20251215 from 20251215_UNet_Myriad2_...)
    datetime_str = None
    if len(parts) > 4:
        datetime_match = re.search(r'^(\d{8})_', parts[4])
        if datetime_match:
            datetime_str = datetime_match.group(1)
    
    return pd.Series({
        'file_path': file_path,
        'training': training,
        'model': model,
        'task': task,
        'n_shots': n_shots,
        'datetime': datetime_str,
        'filename': file_path.split('/')[-1]
    })

# Create DataFrame
df = pd.DataFrame([parse_path(f) for f in pt_files])
df

,file_path,training,model,task,n_shots,datetime,filename
0,finetuning/hydranet/anomaly_detection_nshot100...,finetuning,student,anomaly_detection,1000.0,20251215,UNet_Myriad2_Downstream_unfrozen_best.pt
1,finetuning/hydranet/anomaly_detection_nshot100...,finetuning,student,anomaly_detection,1000.0,20260108,UNet_Myriad2_Downstream_unfrozen_best.pt
2,finetuning/hydranet/anomaly_detection_nshot100...,finetuning,student,anomaly_detection,100.0,20251215,UNet_Myriad2_Downstream_unfrozen_best.pt
3,finetuning/hydranet/anomaly_detection_nshot100...,finetuning,student,anomaly_detection,100.0,20260108,UNet_Myriad2_Downstream_unfrozen_best.pt
4,finetuning/hydranet/anomaly_detection_nshot500...,finetuning,student,anomaly_detection,5000.0,20251215,UNet_Myriad2_Downstream_unfrozen_best.pt
...,...,...,...,...,...,...,...
395,linear_probing/phisatnet/worldfloods_nshot100_...,linear_probing,teacher,worldfloods,100.0,20251213,PhiSatNetDownstream_frozen_best.pt
396,linear_probing/phisatnet/worldfloods_nshot5000...,linear_probing,teacher,worldfloods,5000.0,20251213,PhiSatNetDownstream_frozen_best.pt
397,linear_probing/phisatnet/worldfloods_nshot500_...,linear_probing,teacher,worldfloods,500.0,20251213,PhiSatNetDownstream_frozen_best.pt
398,linear_probing/phisatnet/worldfloods_nshot50_f...,linear_probing,teacher,worldfloods,50.0,20251212,PhiSatNetDownstream_frozen_best.pt


In [ ]:
from huggingface_hub import hf_hub_download

def get_model_weights(training, model, task, n_shots, download_dir=None):
    """
    Get model weights from HuggingFace based on specifications.
    
    Args:
        training: Training type (e.g., 'finetuning', 'linear_probing')
        model: Model name (e.g., 'hydranet', 'phisatnet')
        task: Task name (e.g., 'anomaly_detection', 'worldfloods')
        n_shots: Number of shots (e.g., 50, 100, 500, 1000, 5000)
        download_dir: Directory to save weights (default: HuggingFace cache)
    
    Returns:
        List of local file paths to downloaded weights
    """
    # Query the DataFrame
    results = df[
        (df['training'] == training) &
        (df['model'] == model) &
        (df['task'] == task) &
        (df['n_shots'] == n_shots)
    ]
    
    if len(results) == 0:
        print(f"No weights found for: {training}/{model}/{task}/nshot{n_shots}")
        return []
    
    print(f"Found {len(results)} checkpoint(s)")
    
    # Download files
    local_paths = []
    for _, row in results.iterrows():
        print(f"  Downloading: {row['filename']}")
        local_path = hf_hub_download(
            repo_id=HF_REPO,
            filename=row['file_path'],
            repo_type="dataset",
            local_dir=download_dir
        )
        local_paths.append(local_path)
        print(f"  Saved to: {local_path}")
    
    return local_paths

# Example usage
weights = get_model_weights(
    training='finetuning',
    model='hydranet',
    task='anomaly_detection',
    n_shots=1000,
    download_dir='./weights'
)

In [ ]:
# Save DataFrame to CSV
csv_path = '/Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Paper/src/hydranet/model_weights.csv'
df.to_csv(csv_path, index=False)
print(f"Saved model catalog to: {csv_path}")
print(f"Total records: {len(df)}")

## Test the hydranet package implementation

In [ ]:
# Test the package functionality
# import sys
# sys.path.insert(0, '/Users/roberto.delprete/Library/CloudStorage/OneDrive-ESA/Desktop/Repos/phisat2/Paper/src')

import hydranet

# List available weight combinations
print("Available combinations (first 10):")
print("Note: Model names updated - hydranet->student, phisatnet->teacher")
combos = hydranet.get_available_combinations()
print(combos.head(10))

print("\n" + "="*70)
print("Testing weight download with latest_only=True:")
print("="*70)

# Download weights using the package (downloads only latest)
weights = hydranet.get_model_weights(
    training='finetuning',
    model='student',  # Changed from 'hydranet' to 'student'
    task='anomaly_detection',
    n_shots=100,
    download_dir='../weights',
    latest_only=True
)

print(f"\nDownloaded {len(weights)} weight file(s) (latest only)")

## Test integrated model loading with automatic weights

In [3]:
# Force reload modules
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '..')

# Re-import everything
from src import hydranet
import torch

print("="*70)
print("Example: Loading student model with automatic weights")
print("="*70)

# Load with the 'checkpoint' preset to match HF checkpoint architecture  
# base_filters=16, channel_multipliers=[1, 2, 4, 8]
student_model = hydranet.load_student(
    preset='checkpoint',  # NEW: Matches HF checkpoint architecture
    task='anomaly_detection',
    n_shots=100,
    auto_load_weights=True,
    weights_dir='../weights',
    strict=False  # Allow mismatches for task-specific heads (classifier vs final_conv)
)

print(f"\n✓ Student model loaded successfully!")
print(f"Model type: {type(student_model)}")
print(f"\nModel info:")
print(student_model)

Example: Loading student model with automatic weights
Found latest checkpoint from 20260108
  Downloading: UNet_Myriad2_Downstream_unfrozen_best.pt
  Saved to: ../weights/finetuning/hydranet/anomaly_detection_nshot100_unfrozen/anomaly_detection/20260108_UNet_Myriad2_Downstream_unfrozen_100/UNet_Myriad2_Downstream_unfrozen_best.pt
Loading weights from: ../weights/finetuning/hydranet/anomaly_detection_nshot100_unfrozen/anomaly_detection/20260108_UNet_Myriad2_Downstream_unfrozen_100/UNet_Myriad2_Downstream_unfrozen_best.pt
  Note: 9 keys not found in checkpoint (using random init)
  Note: 2 keys in checkpoint not used (e.g., task-specific heads)

✓ Student model loaded successfully!
Model type: <class 'src.hydranet.models.student.PhisatNet'>

Model info:
PhisatNet(
  depth=3,
  base_filters=16,
  channels=[16, 32, 64, 128],
  parameters=351,155,
  size=1.34 MB
)


In [4]:
# Print detailed layer structure of the loaded student model
print("="*70)
print("PhiSatNet Layer Structure")
print("="*70)
print(student_model)
print("\n" + "="*70)
print("Detailed Layer Breakdown")
print("="*70)

# Print encoders
print("\n[ENCODERS]")
for i, encoder in enumerate(student_model.encoders):
    print(f"\nEncoder {i}:")
    for name, module in encoder.named_children():
        print(f"  {name}: {module}")

# Print bottleneck
print("\n[BOTTLENECK]")
for name, module in student_model.bottleneck.named_children():
    print(f"  {name}: {module}")

# Print upsamplers
print("\n[UPSAMPLERS]")
for i, upsampler in enumerate(student_model.upsamplers):
    print(f"\nUpsampler {i}: {upsampler}")

# Print decoders
print("\n[DECODERS]")
for i, decoder in enumerate(student_model.decoders):
    print(f"\nDecoder {i}:")
    for name, module in decoder.named_children():
        print(f"  {name}: {module}")

# Print final convolution
print("\n[FINAL LAYER]")
print(f"final_conv: {student_model.final_conv}")

# Print parameter counts per component
print("\n" + "="*70)
print("Parameter Count by Component")
print("="*70)
total_params = 0
for name, module in student_model.named_children():
    params = sum(p.numel() for p in module.parameters())
    total_params += params
    print(f"{name:20s}: {params:>10,} parameters")
print(f"{'TOTAL':20s}: {total_params:>10,} parameters")

PhiSatNet Layer Structure
PhisatNet(
  depth=3,
  base_filters=16,
  channels=[16, 32, 64, 128],
  parameters=351,155,
  size=1.34 MB
)

Detailed Layer Breakdown

[ENCODERS]

Encoder 0:
  channel_proj: Conv2d(8, 16, kernel_size=(1, 1), stride=(1, 1))
  convnext_block: ConvNeXtBlock(
  (dwconv): Conv2d(16, 16, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=16)
  (norm): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pwconv1): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1))
  (act): GELU(approximate='none')
  (pwconv2): Conv2d(64, 16, kernel_size=(1, 1), stride=(1, 1))
  (drop_path): Identity()
)

Encoder 1:
  channel_proj: Conv2d(16, 32, kernel_size=(1, 1), stride=(1, 1))
  convnext_block: ConvNeXtBlock(
  (dwconv): Conv2d(32, 32, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=32)
  (norm): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pwconv1): Conv2d(32, 128, kernel_size=(1, 1), stride=(1,